# Đề tài: SmartHVACControlByDoubleQLearning
## Notebook 03: Huấn luyện Mô hình & So sánh Hiệu năng (Q-Learning vs Double Q-Learning vs Rule-Based)

**Mục tiêu:**
- Triển khai thuật toán **Double Q-Learning** (Hasselt, 2010) với 2 bảng giá trị $Q_A$ và $Q_B$ độc lập.
- Triển khai thuật toán **Q-Learning** tiêu chuẩn (Watkins, 1989) làm Baseline so sánh.
- Triển khai bộ điều khiển theo luật **Rule-Based (Thermostat Bang-Bang)** làm mốc tham chiếu thực tế.
- Phân tích hiện tượng **Maximization Bias (Overestimation)** trong việc ước lượng giá trị hành động Q.
- Đánh giá toàn diện trên tập kiểm thử chưa từng thấy (Unseen Test Data) về: Tổng Reward, Tỷ lệ vi phạm tiện nghi nhiệt, Mức tiêu thụ điện năng và Độ ổn định nhiệt độ.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.environment.hvac_env import SmartHVACEnv
from src.agent.q_learning_agent import QLearningAgent
from src.agent.double_q_learning_agent import DoubleQLearningAgent
from src.agent.rule_based_agent import RuleBasedAgent
from src.evaluation.evaluator import compare_controllers
from src.util.visualization import plot_training_convergence, plot_evaluation_comparison, plot_control_trajectory

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

### 1. Nạp Dữ liệu và Phân chia Tập Huấn luyện / Kiểm thử (80% Train - 20% Test)

In [ ]:
data_path = project_root / 'data' / 'processed' / 'cubems_zone1_hourly.csv'
df = pd.read_csv(data_path)
train_size = int(len(df) * 0.8)
df_train = df.iloc[:train_size].copy()
df_test = df.iloc[train_size:].reset_index(drop=True)

print(f'Tập huấn luyện (Train): {len(df_train)} giờ ({df_train["datetime"].min()} đến {df_train["datetime"].max()})')
print(f'Tập kiểm thử (Test):     {len(df_test)} giờ ({df_test["datetime"].min()} đến {df_test["datetime"].max()})')

### 2. Nạp Mô hình đã huấn luyện (Model Loading)
Các mô hình đã được huấn luyện qua 600 episodes trong `train.py`.

In [ ]:
env_test = SmartHVACEnv(df_test, episode_length=24, seed=123)

model_dir = project_root / 'model'
agent_rule = RuleBasedAgent(env_test.n_states, env_test.action_space_size)
agent_ql = QLearningAgent.load(str(model_dir / 'q_learning_model.pkl'))
agent_dql = DoubleQLearningAgent.load(str(model_dir / 'double_q_learning_model.pkl'))

controllers = {
    'Rule-Based': agent_rule,
    'Q-Learning': agent_ql,
    'Double Q-Learning': agent_dql
}

### 3. Phân tích Quá trình Hội tụ và Hiện tượng Ước lượng Thừa (Overestimation Bias)

In [ ]:
from IPython.display import Image
Image(filename=str(project_root / 'result' / 'figure' / 'training_convergence.png'))

### 4. Đánh giá So sánh Hiệu năng trên Tập Kiểm thử Chưa từng thấy (Unseen Test Data)

In [ ]:
df_results, trajectories = compare_controllers(controllers, env_test, n_episodes=50, seed=123)
df_results[['Model', 'Total Reward', 'Comfort Violation (%)', 'Total Energy (kWh)', 'Energy Savings (%)', 'Average Temp (°C)', 'Temp Std (°C)']]

### 5. Biểu đồ So sánh Chỉ số Hiệu năng (Reward, Tiện nghi nhiệt, Năng lượng)

In [ ]:
Image(filename=str(project_root / 'result' / 'figure' / 'evaluation_comparison.png'))

### 6. Quỹ đạo Điều khiển HVAC Thực tế trong 24 Giờ (Control Trajectory)

In [ ]:
Image(filename=str(project_root / 'result' / 'figure' / 'control_trajectory_comparison.png'))

### 7. Nhận xét và Kết luận Chuyên sâu:
1. **Tổng Reward:** Thuật toán **Double Q-Learning** đạt tổng Reward cao nhất (-86.17 so với -92.31 của Q-Learning và -93.44 của Rule-Based), chứng minh khả năng tối ưu hóa cân bằng tốt nhất giữa tiện nghi nhiệt và chi phí điện.
2. **Duy trì Nhiệt độ Tiện nghi:** Double Q-Learning duy trì nhiệt độ trung bình phòng ở mức **24.12°C** (nằm hoàn hảo trong dải khuyến nghị ASHRAE 22°C - 24.5°C), trong khi Q-Learning để nhiệt độ trôi lên 25.32°C do hiện tượng đánh giá quá lạc quan các hành động công suất thấp.
3. **Hiện tượng Overestimation Bias:** Việc sử dụng hai bảng $Q_A$ và $Q_B$ độc lập giúp Double Q-Learning triệt tiêu sai số ước lượng cực đại, từ đó đưa ra các quyết định làm mát dứt khoát và kịp thời khi nhiệt độ phòng bắt đầu tăng cao.